# 837 + 835 → Flattened Claims DataFrame

**Goal:** read a healthcare **837P** file (10 claims — what the provider *bills*),
merge it with an **835** remittance (what the payer *pays*), and produce one tidy
`pandas` DataFrame with **one row per service line**, then save it to CSV.

> **All data here is fabricated. No PHI.** Member IDs, NPIs, and names are invented
> for teaching.

**The big idea for a clinical audience**

| File | Born when | Carries | Format |
|------|-----------|---------|--------|
| **837P** | claim is submitted | codes, charges, provider, patient, dates | X12 EDI (nested segments) |
| **835**  | claim is adjudicated | allowed, paid, patient responsibility, denials | X12 EDI (nested segments) |

Neither is a table. Each is a stream of *segments* ending in `~`, with fields split
by `*`. Our job: parse each into rows, then **join on `claim_id` + `line_number`**.

## Setup

In [1]:
import urllib.request
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

# Input files are read straight from GitHub (raw). Swap these for local paths
# (e.g. Path("sample_837P_10claims.edi")) if you prefer to work offline.
BASE  = "https://raw.githubusercontent.com/thousandoaks/Python4DS-II/refs/heads/main/datasets"
F_837 = f"{BASE}/sample_837P_10claims.edi"
F_835 = f"{BASE}/sample_835_remittance.edi"
OUT   = Path("claims_merged_flattened.csv")

def load_text(src):
    """Read a text file from either an http(s) URL or a local path."""
    src = str(src)
    if src.startswith(("http://", "https://")):
        with urllib.request.urlopen(src) as resp:
            return resp.read().decode("utf-8")
    return Path(src).read_text()

# Split an X12 file into a list of segments, each already split into elements.
# Segments end with '~'; elements are separated by '*'; sub-elements by ':'.
def read_segments(src):
    raw = load_text(src)
    segs = [s.strip() for s in raw.replace("\n", "").split("~") if s.strip()]
    return [s.split("*") for s in segs]

# Peek at the raw 837 so the audience sees what we're parsing
print(load_text(F_837)[:400])

ISA*00*          *00*          *ZZ*SUBMITTER123    *ZZ*RECEIVER456     *260415*1200*^*00501*000000001*0*P*:~
GS*HC*SUBMITTER123*RECEIVER456*20260415*1200*1*X*005010X222A1~
ST*837*0001*005010X222A1~
BHT*0019*00*0123*20260415*1200*CH~
NM1*41*2*SYNTHETIC BILLING SERVICE*****46*SUBMITTER123~
PER*IC*BILLING DEPT*TE*8005551212~
NM1*40*2*SYNTHETIC HEALTH PLAN*****46*RECEIVER456~
HL*1**20*1~
NM1*85*2*RIVE


## Step 1 — Parse the 837 (what was billed)

We walk the segments top to bottom, keeping track of the current billing provider,
subscriber/patient, payer, and claim. Every `SV1` (service line) emits one row.

Key segments:

- `NM1*85` billing provider (+ NPI), `NM1*IL` subscriber, `NM1*PR` payer
- `CLM` claim id + total charge + place of service
- `HI*ABK:...` principal diagnosis (note: **no decimal** — `E11.9` → `E119`)
- `LX` service-line number, `SV1` procedure/charge/units, `DTP*472` service date

In [2]:
def parse_837(path):
    billing_name = billing_npi = member_id = payer = None
    claim_id = pos = primary_dx = None
    line_no = None
    rows = []

    for el in read_segments(path):
        tag = el[0]

        if tag == "NM1":
            entity = el[1]
            if entity == "85":                      # billing provider
                billing_name = el[3]
                if len(el) > 9 and el[8] == "XX":
                    billing_npi = el[9]
            elif entity == "IL" and len(el) > 9:     # subscriber / patient
                member_id = el[9]
            elif entity == "PR":                     # payer
                payer = el[3]

        elif tag == "CLM":
            claim_id = el[1]
            pos = el[5].split(":")[0] if len(el) > 5 else None
            primary_dx = None                        # reset per claim

        elif tag == "HI":                            # diagnoses (take the first)
            primary_dx = el[1].split(":")[1]

        elif tag == "LX":                            # new service line
            line_no = el[1]

        elif tag == "SV1":
            comp = el[1].split(":")                  # HC:code[:modifier]
            proc = comp[1]
            modifier = comp[2] if len(comp) > 2 else ""
            rows.append({
                "claim_id": claim_id,
                "line_number": line_no,
                "member_id": member_id,
                "billing_provider": billing_name,
                "provider_npi": billing_npi,
                "payer_billed": payer,
                "place_of_service": pos,
                "primary_dx": primary_dx,
                "procedure_code": proc,
                "modifier": modifier,
                "charge_amount": float(el[2]),
                "units": el[4] if len(el) > 4 else "",
            })

        elif tag == "DTP" and len(el) > 3 and el[1] == "472":
            rows[-1]["service_date"] = el[3]         # attach date to the last line

    df = pd.DataFrame(rows)
    df["service_date"] = pd.to_datetime(df["service_date"], format="%Y%m%d")
    # make ICD-10 human-readable again: E119 -> E11.9
    df["primary_dx"] = df["primary_dx"].str.replace(r"^([A-Z]\d{2})(\d+)$", r"\1.\2", regex=True)
    return df

billed = parse_837(F_837)
print(billed.shape)
billed.head(8)

(15, 13)


,claim_id,line_number,member_id,billing_provider,provider_npi,payer_billed,place_of_service,primary_dx,procedure_code,modifier,charge_amount,units,service_date
0,CLM2001,1,MBR100001,RIVERSIDE INTERNAL MEDICINE,1990000017,MEDICARE,11,E11.9,99214,,185.0,1,2026-01-14
1,CLM2001,2,MBR100001,RIVERSIDE INTERNAL MEDICINE,1990000017,MEDICARE,11,E11.9,80053,,45.0,1,2026-01-14
2,CLM2001,3,MBR100001,RIVERSIDE INTERNAL MEDICINE,1990000017,MEDICARE,11,E11.9,36415,,15.0,1,2026-01-14
3,CLM2002,1,MBR100002,MAPLE FAMILY PRACTICE,1990000025,BCBS,11,J06.9,99213,,140.0,1,2026-01-20
4,CLM2003,1,MBR100003,HEARTLAND CARDIOLOGY,1990000041,MEDICARE,11,R07.9,99214,,210.0,1,2026-02-03
5,CLM2003,2,MBR100003,HEARTLAND CARDIOLOGY,1990000041,MEDICARE,11,R07.9,93000,,110.0,1,2026-02-03
6,CLM2004,1,MBR100003,HEARTLAND CARDIOLOGY,1990000041,MEDICARE,11,I10,99214,,210.0,1,2026-02-10
7,CLM2005,1,MBR100004,CITY EMERGENCY PHYSICIANS,1990000058,AETNA,23,M54.50,99283,,450.0,1,2026-02-15


## Step 2 — Parse the 835 (what was paid)

Each `CLP` starts a claim (with status, totals, and the *payer's* claim id). Each `SVC`
starts a paid service line. `CAS` carries adjustments — grouped by:

- **CO** = Contractual Obligation (write-off, e.g. reason 45; a denial uses reason 50)
- **PR** = Patient Responsibility (copay / coinsurance / deductible)

`REF*6R` is the line-item control number, which we use as the **line number to join on**.

In [3]:
CLAIM_STATUS = {"1": "Paid (primary)", "2": "Paid (secondary)",
                "3": "Paid (tertiary)", "4": "Denied", "22": "Reversal"}

def parse_835(path):
    claim_id = status = payer_control = filing = patient_id = None
    rows, cur = [], None

    def flush():
        nonlocal cur
        if cur is not None:
            rows.append(cur)
            cur = None

    for el in read_segments(path):
        tag = el[0]

        if tag == "CLP":
            flush()
            claim_id     = el[1]
            status       = el[2]
            payer_control = el[7] if len(el) > 7 else ""
            filing       = el[6] if len(el) > 6 else ""

        elif tag == "NM1" and el[1] == "QC" and len(el) > 9:
            patient_id = el[9]

        elif tag == "SVC":
            flush()
            proc = el[1].split(":")[1]
            cur = {
                "claim_id": claim_id,
                "line_number": None,             # filled from REF*6R
                "claim_status": CLAIM_STATUS.get(status, status),
                "payer_claim_control": payer_control,
                "filing_indicator": filing,
                "patient_id": patient_id,
                "procedure_code": proc,
                "line_charge": float(el[2]),
                "paid_amount": float(el[3]),
                "allowed_amount": 0.0,
                "contractual_adj": 0.0,          # CO group
                "patient_resp": 0.0,             # PR group
                "other_adj": 0.0,
            }

        elif tag == "CAS" and cur is not None:
            group = el[1]
            i = 2
            while i + 1 < len(el):               # reason/amount(/qty) triplets
                try:
                    amt = float(el[i + 1])
                except ValueError:
                    amt = 0.0
                bucket = {"CO": "contractual_adj", "PR": "patient_resp"}.get(group, "other_adj")
                cur[bucket] += amt
                i += 3

        elif tag == "AMT" and cur is not None and el[1] == "B6":
            cur["allowed_amount"] = float(el[2])

        elif tag == "REF" and cur is not None and el[1] == "6R":
            cur["line_number"] = el[2]

    flush()
    return pd.DataFrame(rows)

paid = parse_835(F_835)
print(paid.shape)
paid.head(8)

(15, 13)


,claim_id,line_number,claim_status,payer_claim_control,filing_indicator,patient_id,procedure_code,line_charge,paid_amount,allowed_amount,contractual_adj,patient_resp,other_adj
0,CLM2001,1,Paid (primary),PAYC2001,MB,MBR100001,99214,185.0,96.0,120.0,65.0,24.0,0.0
1,CLM2001,2,Paid (primary),PAYC2001,MB,MBR100001,80053,45.0,11.6,14.5,30.5,2.9,0.0
2,CLM2001,3,Paid (primary),PAYC2001,MB,MBR100001,36415,15.0,3.0,3.0,12.0,0.0,0.0
3,CLM2002,1,Paid (primary),PAYC2002,CI,MBR100002,99213,140.0,76.0,95.0,45.0,19.0,0.0
4,CLM2003,1,Paid (primary),PAYC2003,MB,MBR100003,99214,210.0,108.0,135.0,75.0,27.0,0.0
5,CLM2003,2,Paid (primary),PAYC2003,MB,MBR100003,93000,110.0,28.0,28.0,82.0,0.0,0.0
6,CLM2004,1,Paid (primary),PAYC2004,MB,MBR100003,99214,210.0,108.0,135.0,75.0,27.0,0.0
7,CLM2005,1,Paid (primary),PAYC2005,CI,MBR100004,99283,450.0,168.0,210.0,240.0,42.0,0.0


## Step 3 — Merge billed (837) with paid (835)

The join keys are **`claim_id`** (the `CLM01` in the 837 equals the `CLP01` in the 835)
and **`line_number`** (`LX` in the 837, `REF*6R` in the 835). We keep every billed line
with a `left` join, so a line that never came back on a remittance would surface as a gap.

In [4]:
keys = ["claim_id", "line_number"]

# procedure_code appears in both; keep the 837's and drop the 835 copy after checking
paid_cols = [c for c in paid.columns if c not in ("procedure_code",)]

merged = billed.merge(paid[paid_cols], on=keys, how="left", indicator=True)

print("merge result:", dict(merged["_merge"].value_counts()))
assert (merged["_merge"] == "both").all(), "Some billed lines had no remittance!"
merged = merged.drop(columns="_merge")
merged.head(8)

merge result: {'both': np.int64(15), 'left_only': np.int64(0), 'right_only': np.int64(0)}


,claim_id,line_number,member_id,billing_provider,provider_npi,payer_billed,place_of_service,primary_dx,procedure_code,modifier,charge_amount,units,service_date,claim_status,payer_claim_control,filing_indicator,patient_id,line_charge,paid_amount,allowed_amount,contractual_adj,patient_resp,other_adj
0,CLM2001,1,MBR100001,RIVERSIDE INTERNAL MEDICINE,1990000017,MEDICARE,11,E11.9,99214,,185.0,1,2026-01-14,Paid (primary),PAYC2001,MB,MBR100001,185.0,96.0,120.0,65.0,24.0,0.0
1,CLM2001,2,MBR100001,RIVERSIDE INTERNAL MEDICINE,1990000017,MEDICARE,11,E11.9,80053,,45.0,1,2026-01-14,Paid (primary),PAYC2001,MB,MBR100001,45.0,11.6,14.5,30.5,2.9,0.0
2,CLM2001,3,MBR100001,RIVERSIDE INTERNAL MEDICINE,1990000017,MEDICARE,11,E11.9,36415,,15.0,1,2026-01-14,Paid (primary),PAYC2001,MB,MBR100001,15.0,3.0,3.0,12.0,0.0,0.0
3,CLM2002,1,MBR100002,MAPLE FAMILY PRACTICE,1990000025,BCBS,11,J06.9,99213,,140.0,1,2026-01-20,Paid (primary),PAYC2002,CI,MBR100002,140.0,76.0,95.0,45.0,19.0,0.0
4,CLM2003,1,MBR100003,HEARTLAND CARDIOLOGY,1990000041,MEDICARE,11,R07.9,99214,,210.0,1,2026-02-03,Paid (primary),PAYC2003,MB,MBR100003,210.0,108.0,135.0,75.0,27.0,0.0
5,CLM2003,2,MBR100003,HEARTLAND CARDIOLOGY,1990000041,MEDICARE,11,R07.9,93000,,110.0,1,2026-02-03,Paid (primary),PAYC2003,MB,MBR100003,110.0,28.0,28.0,82.0,0.0,0.0
6,CLM2004,1,MBR100003,HEARTLAND CARDIOLOGY,1990000041,MEDICARE,11,I10,99214,,210.0,1,2026-02-10,Paid (primary),PAYC2004,MB,MBR100003,210.0,108.0,135.0,75.0,27.0,0.0
7,CLM2005,1,MBR100004,CITY EMERGENCY PHYSICIANS,1990000058,AETNA,23,M54.50,99283,,450.0,1,2026-02-15,Paid (primary),PAYC2005,CI,MBR100004,450.0,168.0,210.0,240.0,42.0,0.0


## Step 4 — Flatten, add derived fields, and reconcile

We order the columns, then verify the money adds up on every line:

`charge = paid + patient_resp + contractual_adj + other_adj`

For a denied line the whole charge lands in a CO adjustment (paid and patient_resp are 0).

In [5]:
merged["is_denied"] = merged["claim_status"].eq("Denied")

# Reconciliation check — should be True for every row
recon = (merged["charge_amount"]
         - merged[["paid_amount", "patient_resp", "contractual_adj", "other_adj"]].sum(axis=1))
merged["reconciles"] = recon.abs() < 0.005
print("all lines reconcile:", bool(merged["reconciles"].all()))

col_order = [
    "claim_id", "line_number", "claim_status", "is_denied",
    "member_id", "patient_id", "payer_billed", "filing_indicator",
    "billing_provider", "provider_npi", "place_of_service", "service_date",
    "primary_dx", "procedure_code", "modifier", "units",
    "charge_amount", "allowed_amount", "paid_amount",
    "patient_resp", "contractual_adj", "other_adj",
    "payer_claim_control", "reconciles",
]
flat = merged[col_order].sort_values(["claim_id", "line_number"]).reset_index(drop=True)
flat

all lines reconcile: True


,claim_id,line_number,claim_status,is_denied,member_id,patient_id,payer_billed,filing_indicator,billing_provider,provider_npi,place_of_service,service_date,primary_dx,procedure_code,modifier,units,charge_amount,allowed_amount,paid_amount,patient_resp,contractual_adj,other_adj,payer_claim_control,reconciles
0,CLM2001,1,Paid (primary),False,MBR100001,MBR100001,MEDICARE,MB,RIVERSIDE INTERNAL MEDICINE,1990000017,11,2026-01-14,E11.9,99214,,1,185.0,120.0,96.0,24.0,65.0,0.0,PAYC2001,True
1,CLM2001,2,Paid (primary),False,MBR100001,MBR100001,MEDICARE,MB,RIVERSIDE INTERNAL MEDICINE,1990000017,11,2026-01-14,E11.9,80053,,1,45.0,14.5,11.6,2.9,30.5,0.0,PAYC2001,True
2,CLM2001,3,Paid (primary),False,MBR100001,MBR100001,MEDICARE,MB,RIVERSIDE INTERNAL MEDICINE,1990000017,11,2026-01-14,E11.9,36415,,1,15.0,3.0,3.0,0.0,12.0,0.0,PAYC2001,True
3,CLM2002,1,Paid (primary),False,MBR100002,MBR100002,BCBS,CI,MAPLE FAMILY PRACTICE,1990000025,11,2026-01-20,J06.9,99213,,1,140.0,95.0,76.0,19.0,45.0,0.0,PAYC2002,True
4,CLM2003,1,Paid (primary),False,MBR100003,MBR100003,MEDICARE,MB,HEARTLAND CARDIOLOGY,1990000041,11,2026-02-03,R07.9,99214,,1,210.0,135.0,108.0,27.0,75.0,0.0,PAYC2003,True
5,CLM2003,2,Paid (primary),False,MBR100003,MBR100003,MEDICARE,MB,HEARTLAND CARDIOLOGY,1990000041,11,2026-02-03,R07.9,93000,,1,110.0,28.0,28.0,0.0,82.0,0.0,PAYC2003,True
6,CLM2004,1,Paid (primary),False,MBR100003,MBR100003,MEDICARE,MB,HEARTLAND CARDIOLOGY,1990000041,11,2026-02-10,I10,99214,,1,210.0,135.0,108.0,27.0,75.0,0.0,PAYC2004,True
7,CLM2005,1,Paid (primary),False,MBR100004,MBR100004,AETNA,CI,CITY EMERGENCY PHYSICIANS,1990000058,23,2026-02-15,M54.50,99283,,1,450.0,210.0,168.0,42.0,240.0,0.0,PAYC2005,True
8,CLM2006,1,Denied,True,MBR100005,MBR100005,MEDICAID,MC,MAPLE FAMILY PRACTICE,1990000025,11,2026-03-01,E78.5,99213,,1,140.0,0.0,0.0,0.0,140.0,0.0,PAYC2006,True
9,CLM2007,1,Paid (primary),False,MBR100001,MBR100001,MEDICARE,MB,RIVERSIDE INTERNAL MEDICINE,1990000017,11,2026-03-12,Z00.00,99396,,1,250.0,175.0,175.0,0.0,75.0,0.0,PAYC2007,True


## Step 5 — Save to CSV

In [6]:
flat.to_csv(OUT, index=False)
print(f"Wrote {OUT}  ({len(flat)} rows, {flat['claim_id'].nunique()} claims)")

# A couple of quick analytics the audience can relate to
print("\nPayment rate by payer:")
print(flat.groupby("payer_billed")[["charge_amount", "paid_amount"]].sum())

print("\nDenied lines:")
print(flat.loc[flat["is_denied"], ["claim_id", "procedure_code", "charge_amount"]])

Wrote claims_merged_flattened.csv  (15 rows, 10 claims)

Payment rate by payer:
              charge_amount  paid_amount
payer_billed                            
AETNA                 725.0        308.0
BCBS                  370.0        236.0
MEDICAID              140.0          0.0
MEDICARE             1345.0        660.0

Denied lines:
  claim_id procedure_code  charge_amount
8  CLM2006          99213          140.0


### What to point out in the seminar

- The **837** and **835** are two views of the same claim; the merge is where "what we
  billed" meets "what we got paid."
- Join on **claim id + line number** — always confirm both sides use the same line key.
- The 835 adds columns the 837 never had: `allowed_amount`, `paid_amount`, `patient_resp`,
  and the adjustment/denial reasons.
- Always **reconcile** (`charge = paid + patient_resp + adjustments`) — it's the fastest way
  to catch a parsing bug.

**Real-world caveats (simplified here for teaching):** production files use a proper EDI
parser (e.g. `pyx12`) rather than string splitting; a remittance is issued per payer-payee,
so you often receive several 835s and concatenate them; and 837I (institutional) claims use
different segments than the 837P shown here.